# 第8章: 感情分析への機械学習の応用

この Notebook は、原本 `machine-learning-book/ch08/ch08.ipynb` を最新の Python パッケージ環境と
CI 上の `pytest --nbmake` 実行に合わせて移行したものです。

原本の教育意図を保ちながら、以下の点を現行環境向けに調整しています。

- ネットワークダウンロードや対話処理を省き、読み取り専用サブモジュール内の `movie_data.csv.gz` を参照する
- `nltk.download()` や `pyprind` に依存せず、標準ライブラリと scikit-learn で再現できる構成にする
- ロジスティック回帰、オンライン学習、LDA を CI で現実的な時間に収まるサブセットで検証する


## この Notebook で確認すること

- 原本アセットと IMDb レビューデータを、`src/` 配下から壊れずに参照できることを確認する
- `CountVectorizer` と `TfidfTransformer` による bag-of-words / TF-IDF の基本動作を確認する
- テキスト前処理とトークナイズの流れを再現する
- `TfidfVectorizer` + `LogisticRegression` による感情分類を最新版 scikit-learn で検証する
- `HashingVectorizer` + `SGDClassifier` の out-of-core 学習と LDA によるトピック抽出を軽量構成で実行する


In [ ]:
from importlib.metadata import version
from pathlib import Path
import platform
import re
import sys

from IPython.display import Image, display
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import (
    CountVectorizer,
    ENGLISH_STOP_WORDS,
    HashingVectorizer,
    TfidfTransformer,
    TfidfVectorizer,
)
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "machine-learning-book").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("リポジトリルートを見つけられませんでした。")


REPO_ROOT = find_repo_root()
CHAPTER_DIR = REPO_ROOT / "machine-learning-book" / "ch08"
DATA_PATH = CHAPTER_DIR / "movie_data.csv.gz"
FIGURE_PATH = CHAPTER_DIR / "figures" / "08_1.png"

assert DATA_PATH.exists(), f"データファイルが見つかりません: {DATA_PATH}"
assert FIGURE_PATH.exists(), f"図版ファイルが見つかりません: {FIGURE_PATH}"

print(f"Python 実行ファイル: {sys.executable}")
print(f"Python バージョン: {platform.python_version()}")
print(f"Matplotlib バックエンド: {matplotlib.get_backend()}")
print(f"Chapter directory: {CHAPTER_DIR}")


In [ ]:
package_versions = pd.DataFrame(
    [
        ("numpy", version("numpy")),
        ("pandas", version("pandas")),
        ("matplotlib", version("matplotlib")),
        ("scikit-learn", version("scikit-learn")),
        ("pytest", version("pytest")),
    ],
    columns=["パッケージ", "バージョン"],
)
package_versions


## 原本図版の参照

移行後の Notebook は `src/ch08/` 配下にありますが、書籍の図版とデータは読み取り専用の
`machine-learning-book/` サブモジュールに残しています。ここでは章の代表図版を読み込めることを確認します。


In [ ]:
display(Image(filename=str(FIGURE_PATH), width=700))


## IMDb データの読み込み

原本では生の IMDb ディレクトリから `movie_data.csv` を組み立てていましたが、CI ではその前処理を毎回再実行すると重くなります。
移行版では原本に同梱されている `movie_data.csv.gz` を直接読み込み、統計量とラベル分布を確認します。


In [ ]:
df = pd.read_csv(DATA_PATH, compression="gzip")
df = df.rename(columns={"0": "review", "1": "sentiment"})

summary = pd.Series(
    {
        "レビュー件数": len(df),
        "列数": df.shape[1],
        "positive 件数": int((df["sentiment"] == 1).sum()),
        "negative 件数": int((df["sentiment"] == 0).sum()),
        "平均レビュー長": round(df["review"].str.len().mean(), 1),
    }
)

display(summary.to_frame(name="値"))
display(df.head(3))

fig, ax = plt.subplots(figsize=(4.8, 3.0))
df["sentiment"].value_counts().sort_index().plot(kind="bar", ax=ax, color=["#C44E52", "#4C72B0"])
ax.set_xticklabels(["negative", "positive"], rotation=0)
ax.set_ylabel("Count")
ax.set_title("IMDb Label Distribution")
plt.tight_layout()
plt.show()
plt.close(fig)


## Bag-of-Words の確認

原本と同じ 3 文を使い、`CountVectorizer` によって語彙辞書と出現回数ベクトルがどのように作られるかを確認します。


In [ ]:
docs = np.array(
    [
        "The sun is shining",
        "The weather is sweet",
        "The sun is shining, the weather is sweet, and one and one is two",
    ]
)

count = CountVectorizer()
bag = count.fit_transform(docs)

display(pd.Series(count.vocabulary_).sort_values().to_frame(name="index"))
display(pd.DataFrame(bag.toarray(), columns=count.get_feature_names_out()))


## TF-IDF の確認

`TfidfTransformer` による重み付けと、原本で解説していた単語 `is` の計算例を最新版 scikit-learn の式に合わせて再現します。


In [ ]:
np.set_printoptions(precision=3, suppress=True)

tfidf = TfidfTransformer(use_idf=True, norm="l2", smooth_idf=True)
tfidf_dense = tfidf.fit_transform(bag).toarray()

tf_is = 3
n_docs = len(docs)
df_is = 3
idf_is = np.log((1 + n_docs) / (1 + df_is))
raw_tfidf_is = tf_is * (idf_is + 1)

raw_tfidf = TfidfTransformer(use_idf=True, norm=None, smooth_idf=True).fit_transform(bag).toarray()[-1]
l2_tfidf = raw_tfidf / np.sqrt(np.sum(raw_tfidf**2))

display(pd.DataFrame(tfidf_dense, columns=count.get_feature_names_out()))
pd.Series(
    {
        'manual_raw_tfidf("is")': round(raw_tfidf_is, 3),
        'manual_l2_tfidf("is")': round(l2_tfidf[count.vocabulary_["is"]], 3),
        'sklearn_l2_tfidf("is")': round(tfidf_dense[-1, count.vocabulary_["is"]], 3),
    }
)


## テキスト前処理とトークナイズ

HTML タグ除去、記号正規化、顔文字保持という原本の考え方を引き継ぎつつ、停止語は `nltk` ではなく
scikit-learn の `ENGLISH_STOP_WORDS` を使います。


In [ ]:
def preprocessor(text: str) -> str:
    text = re.sub(r"<[^>]*>", "", text)
    emoticons = re.findall(r"(?::|;|=)(?:-)?(?:\)|\(|D|P)", text)
    text = re.sub(r"[\W]+", " ", text.lower())
    return (text + " " + " ".join(emoticons).replace("-", "")).strip()


def tokenizer(text: str) -> list[str]:
    return text.split()


def tokenizer_no_stopwords(text: str) -> list[str]:
    return [token for token in text.split() if token not in ENGLISH_STOP_WORDS]


sample_text = df.loc[0, "review"][-200:]
clean_text = preprocessor(sample_text)

display(
    pd.Series(
        {
            "original_tail": sample_text,
            "cleaned_tail": clean_text,
            "tokenizer": tokenizer("runners like running and thus they run"),
            "tokenizer_no_stopwords": tokenizer_no_stopwords(
                preprocessor("A runner likes running and runs a lot")
            ),
        }
    ).to_frame(name="値")
)


## ロジスティック回帰による感情分類

原本では 50,000 件全文に対して大きめのグリッドサーチを行っていましたが、CI では重すぎます。
ここでは 6,000 件の層化サンプルを使い、`TfidfVectorizer` と `LogisticRegression` の基本パイプラインを検証します。


In [ ]:
sample_df = df.sample(n=6000, random_state=1).reset_index(drop=True)
sample_df["review_clean"] = sample_df["review"].map(preprocessor)

X_train, X_test, y_train, y_test = train_test_split(
    sample_df["review_clean"],
    sample_df["sentiment"],
    test_size=0.2,
    random_state=1,
    stratify=sample_df["sentiment"],
)

sentiment_clf = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                preprocessor=None,
                tokenizer=str.split,
                token_pattern=None,
                stop_words=list(ENGLISH_STOP_WORDS),
                max_features=10000,
            ),
        ),
        ("clf", LogisticRegression(solver="liblinear", max_iter=1000, random_state=1)),
    ]
)

sentiment_clf.fit(X_train, y_train)
y_pred = sentiment_clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

vectorizer = sentiment_clf.named_steps["tfidf"]
classifier = sentiment_clf.named_steps["clf"]
feature_names = vectorizer.get_feature_names_out()
coef = classifier.coef_[0]

top_positive = pd.Series(coef, index=feature_names).sort_values(ascending=False).head(10)
top_negative = pd.Series(coef, index=feature_names).sort_values().head(10)

display(
    pd.Series(
        {
            "train_size": len(X_train),
            "test_size": len(X_test),
            "test_accuracy": round(accuracy, 4),
        }
    ).to_frame(name="値")
)

top_tokens = pd.DataFrame(
    {
        "positive_words": top_positive.index,
        "positive_coef": np.round(top_positive.values, 3),
        "negative_words": top_negative.index,
        "negative_coef": np.round(top_negative.values, 3),
    }
)
display(top_tokens)


## Out-of-Core 学習

原本の `HashingVectorizer` と `SGDClassifier` によるオンライン学習も残します。
ただし、ストリーミング元は CSV ファイルではなく、同じサンプルデータをミニバッチに分けて扱い、`loss='log_loss'` を使って現行 API に合わせます。


In [ ]:
def iter_minibatches(frame: pd.DataFrame, batch_size: int):
    for start in range(0, len(frame), batch_size):
        batch = frame.iloc[start : start + batch_size]
        if batch.empty:
            continue
        yield batch["review_clean"].tolist(), batch["sentiment"].to_numpy()


hashing = HashingVectorizer(
    decode_error="ignore",
    n_features=2**18,
    alternate_sign=False,
    preprocessor=None,
    tokenizer=str.split,
    token_pattern=None,
)
sgd = SGDClassifier(loss="log_loss", random_state=1)
classes = np.array([0, 1])

stream_train = sample_df.iloc[:2000].copy()
stream_test = sample_df.iloc[2000:2500].copy()

for batch_docs, batch_y in iter_minibatches(stream_train, batch_size=250):
    sgd.partial_fit(hashing.transform(batch_docs), batch_y, classes=classes)

online_accuracy = sgd.score(
    hashing.transform(stream_test["review_clean"].tolist()),
    stream_test["sentiment"].to_numpy(),
)

pd.Series(
    {
        "stream_train_size": len(stream_train),
        "stream_test_size": len(stream_test),
        "online_accuracy": round(float(online_accuracy), 4),
    }
).to_frame(name="値")


## LDA によるトピック抽出

原本の LDA セクションも残し、語彙数と文書数を絞ってトピック上位語を確認します。
感情分類とは別に、レビュー集合の中にどのような潜在トピックがあるかを観察します。


In [ ]:
lda_reviews = sample_df["review_clean"].iloc[:2000]
lda_vectorizer = CountVectorizer(stop_words="english", max_df=0.95, min_df=5, max_features=1000)
X_lda = lda_vectorizer.fit_transform(lda_reviews)

lda = LatentDirichletAllocation(
    n_components=4,
    max_iter=5,
    learning_method="batch",
    random_state=1,
)
topic_matrix = lda.fit_transform(X_lda)

feature_names = lda_vectorizer.get_feature_names_out()
topic_rows = []
for topic_idx, topic in enumerate(lda.components_, start=1):
    top_terms = [feature_names[i] for i in topic.argsort()[:-7:-1]]
    topic_rows.append({"topic": topic_idx, "top_terms": ", ".join(top_terms)})

topic_summary = pd.DataFrame(topic_rows)
dominant_topic = topic_matrix[:, 0].argsort()[::-1][:3]
example_reviews = [
    sample_df.loc[idx, "review"][:220].replace("\n", " ") + " ..."
    for idx in dominant_topic
]

display(topic_summary)
pd.Series(
    {
        "lda_documents": X_lda.shape[0],
        "lda_vocabulary_size": X_lda.shape[1],
        "topic_1_examples": example_reviews,
    }
).to_frame(name="値")


## まとめ

この移行版 Notebook では、第8章の中心となるテキスト前処理、Bag-of-Words、TF-IDF、感情分類、
オンライン学習、LDA を最新の scikit-learn API で再構成しました。

原本 `machine-learning-book/` 配下は変更せず、`src/ch08/ch08.ipynb` から読み取り専用データと図版を参照する構成にしているため、
CI 上でも継続的に章の主要ロジックを検証できます。
